In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from experiments.concept_detector_optimal import (
    DEVICE, load, layer_name, ZeroChannelsHook, capture_block_inputs,
    make_cached_forward, greedy_order, prob_curve, dapc_of)

MODEL_KEY = "vit_small_funny_birds"   # or "vit_base_imagenet"
model, ds, normalize, D, picks, handles, head_dim = load(MODEL_KEY, concept="head")

In [ ]:
# High-variance heuristic (optimal) MoRF/LeRF combos. IMG indexes `picks` (16 bench imgs, seed 0).
#
# vit_small_funny_birds:
#   value b9  : high 12,1  | low 11,8     <- top variance (std .21)
#   qk    b9  : high 9,1   | low 11,8
#   qk    b6  : high 5,1   | low 14,7
#   qk    b10 : high 12,2  | low 6,11
#   value b10 : high 1,2   | low 4,11
#
# vit_base_imagenet:
#   value b11 : high 3,1   | low 14,4     <- negative-DAPC anomaly (std .17)
#   value b7  : high 3,7   | low 5,4
#   value b9  : high 1,7   | low 4,8
#   qk    b9  : high 1,7   | low 4,10
#   qk    b7  : high 1,7   | low 4,12
SITE  = "value"
BLOCK = 9
IMG   = 12

In [ ]:
idx, pred, _ = picks[IMG]
xn = normalize(ds[idx][0].unsqueeze(0)).to(DEVICE)
hook = ZeroChannelsHook(head_dim=head_dim)
h = model.get_submodule(layer_name(SITE, BLOCK)).register_forward_hook(hook)
try:
    cache = capture_block_inputs(model, xn)
    fwd = make_cached_forward(model, cache, BLOCK)
    order = greedy_order(model, xn, pred, hook, D, forward_fn=fwd)
    ot = torch.from_numpy(order).long()
    morf = prob_curve(model, xn, pred, ot, hook, D, fwd)
    lerf = prob_curve(model, xn, pred, ot.flip(0), hook, D, fwd)
finally:
    h.remove()
dapc = dapc_of(morf, lerf)

In [ ]:
xr = np.arange(len(morf))
plt.figure(figsize=(4.2, 3.2))
plt.plot(xr, morf, "r.-", label="MoRF")
plt.plot(xr, lerf, "b.-", label="LeRF")
plt.ylim(-0.02, 1.02)
plt.xlabel("heads removed"); plt.ylabel("p(pred)")
plt.title(f"{MODEL_KEY} {SITE} b{BLOCK} img#{IMG} (ds {idx}) dapc={dapc:+.3f}")
plt.legend(); plt.tight_layout(); plt.show()